# Etching Rate（蝕刻速率）

和軟體用的是同一份計算（`magic/etching/calc.py`）。**平常建議用軟體的「＋ 新增紀錄」**：會先預覽、按確認才寫入、重複會提醒。

這裡適合：一次補登多筆、或自己做其他比較圖。紀錄 CSV 的位置在軟體的 ⚙ 設定 → 蝕刻紀錄。


In [ ]:
%matplotlib inline
import os, sys
sys.path.insert(0, os.path.abspath(".."))
from magic.core import settings as S
from magic.core.plotstyle import apply
from magic.etching.calc import compute_rates, make_row
from magic.etching.store import append_csv, find_duplicates, read_log
from magic.runner import run
apply()
CSV = S.etching_csv()          # 和軟體同一份紀錄
print("紀錄檔：", CSV)

## 計算（先看結果，確認後再寫入）

In [ ]:
inputs = dict(hbn_before=122.8, hbn_after=92.7, t=30,
              marker_before=59.3, marker_after=59.3, sio2_before=None, sio2_after=None)
rates = compute_rates(**inputs, reference="auto")
rates

In [ ]:
WRITE = False      # ← 確認上面的數字沒問題再改成 True
row = make_row(rates, **inputs, method="ICP", power="60W", bias="5V", pressure="1Pa",
               sample_type="direct", name="", gas="SF6")
dup = find_duplicates(read_log(CSV), row)
print("和這幾筆重複：", dup) if dup else None
if WRITE and not dup:
    append_csv(row, CSV)
    print("已寫入")

## 比較（篩選取代原本寫死的 row_indices）

In [ ]:
df = read_log(CSV)
if df.empty:
    from magic.testdata import etching_csv
    demo = os.path.join(str(S.data_dir()), "demo", "Etching", "etch_rate_log.csv")
    etching_csv(demo)
    df = read_log(demo)
    print("紀錄是空的，先用範例資料（合成）")
sel = df[(df["Method"] == "ICP")]             # ← 例：只看 ICP
fig, res = run("Etching", "條件比較（平均 ± 標準差）", [sel], by="Power", by2="Gas")
print(res.text)
fig